# Optimización definitiva de Crop6 con CVAE β

Optuna ajusta gamma, bias y slope con estudios separados para acc_both y SSIM en Fashion-MNIST y MNIST. Ambos CVAE usan dimensión intermedia 256 y dimensión latente 64. Se carga el CVAE entrenado con β=0.4 para Fashion-MNIST y con β=0.6 para MNIST.

Cada estudio usa 100 pruebas de 30 iteraciones con un lote fijo de 1.000 mezclas de entrenamiento y semilla 3333. Los rangos son gamma de 0 a 2, bias de 0.05 a 0.45 y slope de 5 a 60 en escala logarítmica.

Los estudios se guardan en `results_crop6_beta/<dataset>/<objetivo>/Crop6.sqlite3`. Al volver a ejecutar un estudio, se completan las pruebas pendientes.

## Configuración y evaluación

In [1]:
from pathlib import Path
import sys

import numpy as np
import optuna

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "project" / "CROP" / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abrí el notebook desde el repositorio.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENTS_DIR = PROJECT_ROOT / "project" / "definitive_experiments"
RESULTS_DIR = EXPERIMENTS_DIR / "results_crop6_beta"
SEED = 3333
LAT, INTER = 64, 256
N_IMAGES = 1000
N_TRIALS = 100
STUDY_ITERATIONS = 30

DATASETS = {"fashion": "Fashion-MNIST", "mnist": "MNIST"}
CVAE_NAMES = {
    "fashion": f"int_{INTER}_lat_{LAT}_beta_0_4_fashion.keras",
    "mnist": f"int_{INTER}_lat_{LAT}_beta_0_6_mnist.keras",
}
SEARCH_SPACE = {
    "gamma": [0.0, 2.0, False],
    "bias": [0.05, 0.45, False],
    "slope": [5.0, 60.0, True],
}
FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "alpha_mix": 0.5,
    "beta": 1,
}


def result_dir(dataset, objective_metric):
    return RESULTS_DIR / dataset / objective_metric


def load_inputs(dataset, cvae_name):
    import tensorflow as tf
    from project.CROP.models.crop6 import Crop6
    from project.loader import Loader

    tf.keras.utils.set_random_seed(SEED)
    return (
        Loader.data(dataset),
        Loader.cvae(model_name=cvae_name),
        Loader.predictor(dataset=dataset),
        Crop6,
    )


def sample_batch(data, split, seed, n_images):
    rng = np.random.default_rng(seed)
    x0, x1 = data[f"x_{split}"], data[f"x_{split}_1"]
    y0, y1 = data[f"y_{split}"], data[f"y_{split}_1"]
    idx0 = rng.choice(len(x0), size=n_images, replace=False)
    idx1 = rng.choice(len(x1), size=n_images, replace=False)
    return x0[idx0], x1[idx1], y0[idx0], y1[idx1]


def evaluate(inputs, batch, params, seed, iterations):
    import tensorflow as tf

    data, cvae, predictor, model_class = inputs
    tf.keras.utils.set_random_seed(seed)
    model = model_class(cvae=cvae, predictor=predictor)
    return model.unmix(
        *batch,
        params=params,
        iterations=iterations,
        labels=data["labels"],
        show_metrics=False,
    )


def scalar_metric(metrics, key):
    value = metrics[key]
    return float(value[0] if isinstance(value, (tuple, list)) else value)

/home/galsan/venvs/tensorflow-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Estudios de Optuna

In [2]:
def study_location(dataset, objective_metric):
    path = result_dir(dataset, objective_metric) / "Crop6.sqlite3"
    name = f"{dataset}_{objective_metric}_Crop6_beta"
    return path, f"sqlite:///{path.as_posix()}", name


def load_study(dataset, objective_metric):
    path, storage, name = study_location(dataset, objective_metric)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el estudio en {path}")
    return optuna.load_study(storage=storage, study_name=name)


def optimize_crop6(dataset, objective_metric, inputs, train_batch):
    path, storage, name = study_location(dataset, objective_metric)
    path.parent.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        storage=storage,
        study_name=name,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        load_if_exists=True,
    )
    config = {
        "seed": SEED,
        "images": N_IMAGES,
        "iterations": STUDY_ITERATIONS,
        "latent_dim": LAT,
        "intermediate_dim": INTER,
        "cvae_name": CVAE_NAMES[dataset],
        "fixed_params": FIXED_PARAMS,
        "search_space": SEARCH_SPACE,
    }
    saved_config = study.user_attrs.get("config")
    if saved_config is None and not study.get_trials(deepcopy=False):
        study.set_user_attr("config", config)
    elif saved_config != config:
        raise ValueError(f"La configuración guardada en {path} difiere de la actual.")

    def objective(trial):
        params = {
            key: trial.suggest_float(key, low, high, log=log)
            for key, (low, high, log) in SEARCH_SPACE.items()
        }
        effective_params = {**FIXED_PARAMS, **params}
        metrics = evaluate(inputs, train_batch, effective_params, SEED, STUDY_ITERATIONS)
        score = scalar_metric(metrics, objective_metric)
        if not np.isfinite(score):
            raise ValueError(f"Métrica no finita en {name}: {score}")
        trial.set_user_attr("effective_params", effective_params)
        for key in ("acc_both", "ssim", "recon_bpsnr"):
            value = scalar_metric(metrics, key)
            trial.set_user_attr(key, value if np.isfinite(value) else None)
        return score

    completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,), deepcopy=False)
    remaining = max(0, N_TRIALS - len(completed))
    if remaining:
        if completed:
            # Evitar repetir las muestras iniciales al retomar el estudio.
            study.sampler = optuna.samplers.TPESampler(seed=SEED + len(completed))
        study.optimize(objective, n_trials=remaining, n_jobs=1)

    print(f"{name} · {study.best_value:.6g} · {study.best_params}", flush=True)
    return study


def plot_study(study, dataset, objective_metric):
    from optuna.visualization import plot_slice

    fig = plot_slice(study, params=list(SEARCH_SPACE),
                     target_name=objective_metric)
    fig.update_layout(title=f"{DATASETS[dataset]} · Crop6 · {objective_metric}")
    fig.show()


def run_experiment(dataset, objective_metric):
    inputs = load_inputs(dataset, CVAE_NAMES[dataset])
    train_batch = sample_batch(inputs[0], "train", SEED, N_IMAGES)
    return optimize_crop6(dataset, objective_metric, inputs, train_batch)

## Fashion-MNIST

In [3]:
fashion_acc = run_experiment("fashion", "acc_both")

I0000 00:00:1795843012.679255  147268 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795843012.729959  147268 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795843014.010432  147268 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Usando fashion como dataset


I0000 00:00:1795843015.513401  147268 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1647 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
[I 2026-11-28 02:16:57,925] A new study created in RDB with name: fashion_acc_both_Crop6_beta
I0000 00:00:1795843018.535654  147268 cuda_dnn.cc:461] Loaded cuDNN version 92801
[I 2026-11-28 02:17:01,948] Trial 0 finished with value: 0.275 and parameters: {'gamma': 1.5031512144089574, 'bias': 0.0937005101554171, 'slope': 16.733352967197288}. Best is trial 0 with value: 0.275.
[I 2026-11-28 02:17:02,995] Trial 1 finished with value: 0.495 and parameters: {'gamma': 0.9996623554960684, 'bias': 0.3128439996079373, 'slope': 8.985295429358263}. Best is trial 1 with value: 0.495.
[I 2026-11-28 02:17:04,282] Trial 2 finished with value: 0.382 and parameters: {'gamma': 1.2255897813211882, 'bias': 0.09786096135897537, 'slope': 29.347111780987284}. Best is trial 1

fashion_acc_both_Crop6_beta · 0.668 · {'gamma': 0.3590700868437654, 'bias': 0.2711379740249488, 'slope': 15.208077909976144}


In [4]:
fashion_ssim = run_experiment("fashion", "ssim")

Usando fashion como dataset


[I 2026-11-28 02:19:17,624] A new study created in RDB with name: fashion_ssim_Crop6_beta
[I 2026-11-28 02:19:19,037] Trial 0 finished with value: 0.568987250328064 and parameters: {'gamma': 1.5031512144089574, 'bias': 0.0937005101554171, 'slope': 16.733352967197288}. Best is trial 0 with value: 0.568987250328064.
[I 2026-11-28 02:19:20,431] Trial 1 finished with value: 0.6354305148124695 and parameters: {'gamma': 0.9996623554960684, 'bias': 0.3128439996079373, 'slope': 8.985295429358263}. Best is trial 1 with value: 0.6354305148124695.
[I 2026-11-28 02:19:21,861] Trial 2 finished with value: 0.5980706214904785 and parameters: {'gamma': 1.2255897813211882, 'bias': 0.09786096135897537, 'slope': 29.347111780987284}. Best is trial 1 with value: 0.6354305148124695.
[I 2026-11-28 02:19:23,354] Trial 3 finished with value: 0.6336711645126343 and parameters: {'gamma': 0.7451210841006557, 'bias': 0.259907309133894, 'slope': 9.618436608383888}. Best is trial 1 with value: 0.6354305148124695.
[I

fashion_ssim_Crop6_beta · 0.665092 · {'gamma': 1.067452300622019, 'bias': 0.2619412758897423, 'slope': 17.89136648036436}


## MNIST

In [5]:
mnist_acc = run_experiment("mnist", "acc_both")

Usando mnist como dataset


[I 2026-11-28 02:21:34,677] A new study created in RDB with name: mnist_acc_both_Crop6_beta
[I 2026-11-28 02:21:36,189] Trial 0 finished with value: 0.622 and parameters: {'gamma': 1.5031512144089574, 'bias': 0.0937005101554171, 'slope': 16.733352967197288}. Best is trial 0 with value: 0.622.
[I 2026-11-28 02:21:37,604] Trial 1 finished with value: 0.846 and parameters: {'gamma': 0.9996623554960684, 'bias': 0.3128439996079373, 'slope': 8.985295429358263}. Best is trial 1 with value: 0.846.
[I 2026-11-28 02:21:38,925] Trial 2 finished with value: 0.771 and parameters: {'gamma': 1.2255897813211882, 'bias': 0.09786096135897537, 'slope': 29.347111780987284}. Best is trial 1 with value: 0.846.
[I 2026-11-28 02:21:40,344] Trial 3 finished with value: 0.873 and parameters: {'gamma': 0.7451210841006557, 'bias': 0.259907309133894, 'slope': 9.618436608383888}. Best is trial 3 with value: 0.873.
[I 2026-11-28 02:21:41,756] Trial 4 finished with value: 0.803 and parameters: {'gamma': 1.24326244520

mnist_acc_both_Crop6_beta · 0.893 · {'gamma': 0.3986099841737827, 'bias': 0.2815082261624183, 'slope': 13.375452320206286}


In [6]:
mnist_ssim = run_experiment("mnist", "ssim")

Usando mnist como dataset


[I 2026-11-28 02:23:53,289] A new study created in RDB with name: mnist_ssim_Crop6_beta
[I 2026-11-28 02:23:54,719] Trial 0 finished with value: 0.6894931793212891 and parameters: {'gamma': 1.5031512144089574, 'bias': 0.0937005101554171, 'slope': 16.733352967197288}. Best is trial 0 with value: 0.6894931793212891.
[I 2026-11-28 02:23:56,126] Trial 1 finished with value: 0.7806023955345154 and parameters: {'gamma': 0.9996623554960684, 'bias': 0.3128439996079373, 'slope': 8.985295429358263}. Best is trial 1 with value: 0.7806023955345154.
[I 2026-11-28 02:23:57,594] Trial 2 finished with value: 0.7607634663581848 and parameters: {'gamma': 1.2255897813211882, 'bias': 0.09786096135897537, 'slope': 29.347111780987284}. Best is trial 1 with value: 0.7806023955345154.
[I 2026-11-28 02:23:58,931] Trial 3 finished with value: 0.7817611694335938 and parameters: {'gamma': 0.7451210841006557, 'bias': 0.259907309133894, 'slope': 9.618436608383888}. Best is trial 3 with value: 0.7817611694335938.
[I

mnist_ssim_Crop6_beta · 0.829895 · {'gamma': 0.7084219157624791, 'bias': 0.2238513386039232, 'slope': 28.204359461949313}


## Resultados guardados y evaluación en prueba

Elegí un estudio guardado para ver sus mejores parámetros y los gráficos de cortes de Optuna. La última celda evalúa esos parámetros en un lote independiente de prueba con la misma cantidad de mezclas e iteraciones del estudio.

In [9]:
dataset, objective_metric = "fashion", "acc_both"

study = load_study(dataset, objective_metric)
print(f"Crop6 · {study.best_value:.6g} · {study.best_params}")
plot_study(study, dataset, objective_metric)

Crop6 · 0.668 · {'gamma': 0.3590700868437654, 'bias': 0.2711379740249488, 'slope': 15.208077909976144}


In [8]:
config = study.user_attrs["config"]
test_seed = config["seed"] + 1
inputs = load_inputs(dataset, config["cvae_name"])
test_batch = sample_batch(inputs[0], "test", test_seed, config["images"])
params = {**config["fixed_params"], **study.best_params}
test_metrics = evaluate(inputs, test_batch, params, test_seed, config["iterations"])

for metric in ("acc_both", "ssim", "recon_bpsnr"):
    print(f"{metric} = {scalar_metric(test_metrics, metric):.6g}")

Usando fashion como dataset
acc_both = 0.588
ssim = 0.667989
recon_bpsnr = 18.3201
